## Imports

In [1]:
import sys
import os
sys.path.append(os.path.expanduser("~/Desktop/Kaitlyn_Catalyst/ct_classifier/speciesnet"))
from speciesnet.classifier import SpeciesNetClassifier
from speciesnet.detector import SpeciesNetDetector
from speciesnet.constants import Detection
from speciesnet.ensemble_prediction_combiner import combine_predictions_for_single_item
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image, ImageDraw
%matplotlib inline
import matplotlib.pyplot as plt
import torch.nn as nn

In [ ]:
# # Get the number of new total classes
# model = SpeciesNetClassifier(model_name=classifier_model_name, target_species_txt=target_species_txt)
# # original_num_classes = len(model.labels)  # Original classes from speciesnet_label.json
# # new_labels = ["Mongoose"]

## Download models

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.model_download("google/speciesnet/pyTorch/v4.0.1a")

# print("Path to model files:", path)

## Initializing

In [6]:
# === Paths ===
image_path = "/Users/sarahabdelazim/Desktop/Kaitlyn_Catalyst/ct_classifier/datasets/all_species_images/496.JPG"
img = Image.open(image_path).convert("RGB")

# === Device Setup ===
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## Detection

In [ ]:
detector_model_name = "/Users/sarahabdelazim/.cache/kagglehub/models/google/speciesnet/pyTorch/v4.0.1a/1"

# === Create an Instance of Detector ===
detector = SpeciesNetDetector(detector_model_name)
preprocessed_image = detector.preprocess(img)

# === Run Detection ===
detections_result = detector.predict(filepath=image_path, img=preprocessed_image)

# === Display Detection Results ===
print("Detections:")
for detection in detections_result.get("detections", []):
    print(
        f"Category: {detection['category']}, "
        f"Label: {detection['label']}, "
        f"Confidence: {detection['conf']:.2f}, "
        f"BBox: {detection['bbox']}"
    )

# === Handle Failures ===
if "failures" in detections_result:
    print("Detection failed for the following reasons:", detections_result["failures"])

draw = ImageDraw.Draw(img)

# Iterate over detections
for detection in detections_result.get("detections", []):
    bbox = detection['bbox']  # [x_min, y_min, width, height]
    
    # Convert normalized bbox (relative coordinates) to absolute pixel values
    x_min = int(bbox[0] * img.width)
    y_min = int(bbox[1] * img.height)
    # Correctly calculate x_max and y_max
    x_max = int((bbox[0] + bbox[2]) * img.width)
    y_max = int((bbox[1] + bbox[3]) * img.height)

    # Debugging: Print bounding box values
    print(f"Normalized bbox: {bbox}")
    print(f"Absolute bbox: x_min={x_min}, y_min={y_min}, x_max={x_max}, y_max={y_max}")

    # Skip invalid bounding boxes
    if x_max < x_min or y_max < y_min:
        print(f"Skipping invalid bbox: x_min={x_min}, y_min={y_min}, x_max={x_max}, y_max={y_max}")
        continue

    # Draw rectangle (bounding box)
    draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=3)

    # Optional: Add label and confidence score
    label = f"{detection['label']} ({detection['conf']:.2f})"
    draw.text((x_min, y_min), label, fill="red")

# === Display Image with Bounding Boxes in Jupyter ===
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')  # Turn off axes for better visualization
plt.show()

## Classification

In [ ]:
# for name, layer in model.model.named_children():
#     print(f"{name}: {layer}")
# for name, layer in model.model.named_modules():
#     print(f"{name}: {layer}")

In [24]:
# # === Configuration ===
# classifier_model_name = "/Users/sarahabdelazim/.cache/kagglehub/models/google/speciesnet/pyTorch/v4.0.1a/1"
# target_species_txt = "/Users/sarahabdelazim/Desktop/Kaitlyn_Catalyst/ct_classifier/target_species.txt"

# device = torch.device("mps" if torch.backends.mps.is_available()
#                       else "cuda" if torch.cuda.is_available()
#                       else "cpu")



# # # === Load base classifier
# classifier = SpeciesNetClassifier(model_name=classifier_model_name, target_species_txt=target_species_txt)
# print(len(classifier.labels))

# # Load image and convert to numpy
# img = Image.open(image_path).convert("RGB")
# img = img.resize((480, 480))  # Resize manually

# arr = np.array(img).astype(np.float32) / 255.0  # [H, W, C]
# x = torch.tensor(arr).unsqueeze(0).to(device)  # [1, H, W, C]

# print("✅ Input shape:", x.shape)  # Should be [1, 480, 480, 3]

# # === Run inference
# with torch.no_grad():
#     logits = classifier.model(x) 
#     target_indices = classifier.target_idx   
#     print(len(target_indices))
#     target_logits = logits[0, target_indices]  # select relevant logits
#     print(len(target_logits))
#     target_probs = F.softmax(target_logits, dim=0)

#     # Get top-5 among filtered classes
#     topk = min(5, len(target_indices))
#     top_probs, top_indices = torch.topk(target_probs, topk)

#     # Combine original target labels + new label
#     filtered_labels = classifier.target_labels 
#     top_classes = [filtered_labels[idx.item()] for idx in top_indices]
#     top_scores = [top_probs[i].item() for i in range(topk)]

# # === Output Top-5
# print("🔝 Top-5 Predictions:")
# for i, (label, score) in enumerate(zip(top_classes, top_scores)):
#     print(f"{i+1}. {label} ({score*100:.2f}%)")

In [79]:
from speciesnet.classifier import SpeciesNetClassifier
from PIL import Image

# === Configuration ===
classifier_model_name = "/Users/sarahabdelazim/.cache/kagglehub/models/google/speciesnet/pyTorch/v4.0.1a/1"
target_species_txt = "/Users/sarahabdelazim/Desktop/Kaitlyn_Catalyst/ct_classifier/target_species.txt"

# Load the classifier
classifier = SpeciesNetClassifier(model_name=classifier_model_name, target_species_txt=target_species_txt)
print(f"✅ Loaded model with {len(classifier.labels)} total classes.")

# === Load and preprocess image using built-in method
img = Image.open(image_path).convert("RGB")
preprocessed = classifier.preprocess(img)

# === Run prediction using built-in method
result = classifier.predict(filepath=image_path, img=preprocessed)

# === Output Top-5 Predictions
if "classifications" in result:
    print("🔝 Top-5 Predictions:")
    for i, (label, score) in enumerate(zip(result["classifications"]["classes"], result["classifications"]["scores"])):
        print(f"{i+1}. {label} ({score*100:.2f}%)")
else:
    print(f"❌ Prediction failed for: {result['filepath']}")
    print(f"Reason: {result.get('failures')}")

✅ Loaded model with 2498 total classes.
🔝 Top-5 Predictions:
1. 7a5a9437-883c-4a5b-b7f9-1d90ec457f46;mammalia;tubulidentata;orycteropodidae;orycteropus;afer;aardvark (90.84%)
2. 568b60c5-123c-4d70-b918-c6ecb7dfb39f;mammalia;cingulata;chlamyphoridae;euphractus;sexcinctus;yellow armadillo (2.09%)
3. d23e77b7-0314-4129-b2e8-810e9c4771e7;mammalia;cingulata;chlamyphoridae;priodontes;maximus;giant armadillo (1.14%)
4. 89d41ec3-af8a-438a-86dd-4a6cb1819f07;mammalia;pilosa;myrmecophagidae;tamandua;tetradactyla;southern tamandua (1.00%)
5. eebbc7c1-32e7-4938-9099-b47ad491b6f9;mammalia;diprotodontia;macropodidae;lagorchestes;conspicillatus;spectacled hare-wallaby (0.78%)


In [102]:
print(type(classifier))
print(dir(classifier.model))  # Look for attributes like .model, .backbone, .classifier, etc.

<class 'speciesnet.classifier.SpeciesNetClassifier'>
['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_se

In [85]:
# for name, module in classifier.model.named_modules():
#     if "dense" in name.lower() or "matmul" in name.lower() or "bias" in name.lower():
#         print(name, ":", type(module))

In [86]:
# dense_layer = dict(classifier.model.named_modules())["SpeciesNet/dense/MatMul"]
# print(dense_layer)

In [ ]:
# outputs = {}

# def get_hook(name):
#     def hook_fn(module, input, output):
#         outputs[name] = output
#     return hook_fn

# # Register hooks
# for name, module in classifier.model.named_modules():
#     if "dense" in name.lower():
#         print("Hooking into layer:", name)
#         module.register_forward_hook(get_hook(name))

Hooking into layer: SpeciesNet/dense/MatMul
Hooking into layer: SpeciesNet/dense/BiasAdd


In [ ]:
# result = classifier.predict(filepath=image_path, img=preprocessed)
# for layer_name, output in outputs.items():
#     print(f"🔍 Output from {layer_name}: {output.shape}")

🔍 Output from SpeciesNet/dense/MatMul: torch.Size([1, 2498])
🔍 Output from SpeciesNet/dense/BiasAdd: torch.Size([1, 2498])


In [ ]:
# matmul_layer = dict(classifier.model.named_modules())["SpeciesNet/dense/MatMul"]
# print("Weights:", list(matmul_layer.parameters()))

Weights: []


In [71]:
# for name, module in classifier.model.named_modules():
#     print(name, ":", type(module))

## Ensemble

In [ ]:
# country = "MOZ"  # Optional country information
# admin1_region = "Sofala"  # Optional region information

# # Maps and configuration
# taxonomy_map = {}  # Define your taxonomy mapping
# geofence_map = {}  # Define your geofence mapping
# enable_geofence = True

# # Define geofencing and roll-up functions
# def geofence_fn(labels, scores, country, admin1_region, taxonomy_map, geofence_map, enable_geofence):
#     # Example geofencing logic
#     return labels[0], scores[0], "geofence"

# def roll_up_fn(labels, scores, country, admin1_region, target_taxonomy_levels, non_blank_threshold, taxonomy_map, geofence_map, enable_geofence):
#     # Example roll-up logic
#     return labels[0], scores[0], "rollup"

# # === Combine Predictions ===
# ensemble_result = combine_predictions_for_single_item(
#     classifications=classifications,
#     detections = detections_result.get("detections", []),
#     country=country,
#     admin1_region=admin1_region,
#     taxonomy_map=taxonomy_map,
#     geofence_map=geofence_map,
#     enable_geofence=enable_geofence,
#     geofence_fn=geofence_fn,
#     roll_up_fn=roll_up_fn
# )

# # === Output the Result ===
# label, score, source = ensemble_result
# print(f"Ensembled Prediction: {label} (Confidence: {score}, Source: {source})")

## Get Layer Names

In [ ]:
# # Retrieve and print the layer names
# layer_names = [name for name, _ in classifier.model.named_modules()]
# print("Layer Names:")
# for name in layer_names:
#     print(name)

In [ ]:
import sys
import os
sys.path.append(os.path.expanduser("~/Desktop/Kaitlyn_Catalyst/ct_classifier/speciesnet"))
from dataloader import SpeciesImageDataset
from speciesnet.classifier import SpeciesNetClassifier
from model import AugmentedSpeciesNet
from torch.utils.data import DataLoader, TensorDataset


base_dir = os.path.expanduser("~/Desktop/Kaitlyn_Catalyst/ct_classifier")
csv_path = os.path.join(base_dir, "notebooks", "full_df_filtered.csv")
target_species_txt = os.path.join(base_dir, "target_species.txt")
image_dir = os.path.join(base_dir, "datasets", "all_species_images")
train_filtered_path = os.path.join(base_dir, "speciesnet/train_filtered.csv")
val_filtered_path = os.path.join(base_dir, "speciesnet/val_filtered.csv")
train_filtered_df = pd.read_csv(train_filtered_path)
val_filtered_df = pd.read_csv(val_filtered_path)
classifier_model_name = os.path.expanduser("~/.cache/kagglehub/models/google/speciesnet/pyTorch/v4.0.1a/1")

classifier = SpeciesNetClassifier(model_name=classifier_model_name, target_species_txt=target_species_txt)
original_outputs = len(classifier.labels)
target_labels = len(classifier.target_labels)
classifier.model = AugmentedSpeciesNet(classifier.model, original_outputs, target_labels)


# === DataLoaders ===
train_dataset = SpeciesImageDataset(train_filtered_df, image_dir, classifier)
val_dataset = SpeciesImageDataset(val_filtered_df, image_dir, classifier)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True, num_workers=0)

In [14]:
print("Number of training samples:", len(train_loader))
print("Number of validation samples:", len(val_loader))

Number of training samples: 12
Number of validation samples: 11
